# 01 - Exploring MeerKAT Visibility Data

**Goal of this notebook:** get comfortable looking at raw MeerKAT visibility
data before we turn it into a machine learning dataset.

You will:
1. Load an observation (real `.rdb` file, or synthetic data for practice).
2. Print basic dataset statistics (dimensions, baselines, channels, duration, frequency range, flagged fraction).
3. Visualize one baseline's amplitude, phase, and AOFlagger mask as "waterfall" plots (time on the y-axis, frequency on the x-axis).

### Background
MeerKAT correlates the signals from every pair of antennas (a *baseline*).
For each baseline, at every time step and frequency channel, we get one
complex number (a *visibility*). RFI (radio frequency interference) shows
up as unusual jumps in amplitude/phase, and MeerKAT's online pipeline
(AOFlagger) has already flagged likely RFI samples for us - those flags
are the ground-truth labels we'll use for machine learning in the next notebook.


## 1. Setup

In [ ]:
import sys
sys.path.append("..")  # so we can import the meerkat_ml package

import numpy as np
import matplotlib.pyplot as plt

from data.loader import load_observation, make_synthetic_observation
from data.preprocessing import extract_amplitude, extract_phase, extract_labels
from data.visualization import plot_baseline_summary

%matplotlib inline


## 2. Load the data

If you have a real MeerKAT `.rdb` file, set `USE_REAL_DATA = True` and point
`RDB_PATH` at it. Otherwise, we'll generate a small synthetic observation
that mimics the same structure (time, frequency, baseline) so you can
follow along without needing real data.


In [ ]:
USE_REAL_DATA = False
RDB_PATH = "meerkat_data.rdb"

if USE_REAL_DATA:
    observation = load_observation(RDB_PATH, pol="HH", corrprods="cross", scans="track")
else:
    print("Using synthetic data (set USE_REAL_DATA = True to load a real .rdb file).")
    observation = make_synthetic_observation(n_time=512, n_freq=512, n_baseline=4)

print("Observation loaded.")


## 3. Basic dataset statistics

In [ ]:
print(f"Visibility cube shape (time, freq, baseline): {observation.vis.shape}")
print(f"Flag cube shape (time, freq, baseline):        {observation.flags.shape}")
print()
print(f"Number of baselines:      {observation.n_baseline}")
print(f"Number of freq channels:  {observation.n_freq}")
print(f"Number of time steps:     {observation.n_time}")
print(f"Observation duration:     {observation.duration_seconds:.1f} seconds")

freq_min, freq_max = observation.freq_range_hz
print(f"Frequency range:          {freq_min/1e6:.2f} MHz - {freq_max/1e6:.2f} MHz")
print(f"Percentage flagged (RFI): {observation.flagged_fraction * 100:.2f}%")


## 4. Visualize one baseline

Let's pick a single baseline and look at its amplitude waterfall, phase
waterfall, and AOFlagger mask side by side.

Remember the convention:
- **x-axis = frequency**
- **y-axis = time**


In [ ]:
BASELINE_INDEX = 0  # try changing this to look at other baselines
print(f"Inspecting baseline: {observation.baseline_labels[BASELINE_INDEX]}")

amplitude = extract_amplitude(observation.vis, BASELINE_INDEX)
phase = extract_phase(observation.vis, BASELINE_INDEX)
labels = extract_labels(observation.flags, BASELINE_INDEX)

print(f"Amplitude range: {amplitude.min():.3f} - {amplitude.max():.3f}")
print(f"Fraction flagged on this baseline: {labels.mean() * 100:.2f}%")


In [ ]:
fig = plot_baseline_summary(
    amplitude, phase, labels,
    baseline_label=observation.baseline_labels[BASELINE_INDEX],
)
plt.show()


## 5. Things to try

- Change `BASELINE_INDEX` above and re-run - does the RFI look the same on every baseline?
- Look closely at the amplitude waterfall: can you spot the RFI streaks *before* looking at the mask?
- What's the difference between how narrow-band and broad-band RFI appear in the waterfall?
- (If using real data) How does the flagged fraction compare across different observations or times of day?

Once you're comfortable with what the data looks like, move on to
**`02_create_dataset.ipynb`** to turn this into a machine-learning-ready
dataset of small image patches.
